# Pretrain exploration from a perception checkpoint

First run `nectary.ipynb`. This notebook loads its perception checkpoint into the
uncertainty-gated model and trains only exploration, with perception and
exploitation learning rates set to zero. The default environment is flat with a
cross. Change `PRETRAIN_ENVIRONMENT` to prepare the
other three control conditions.

In [ ]:
from pathlib import Path
import os
import sys

PACKAGE_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "NectarySearch").is_dir() and (p / "FlowerTracking").is_dir()),
    None,
)
if PACKAGE_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted code folder.")
NOTEBOOK_DIR = PACKAGE_ROOT / "NectarySearch"
sys.path.insert(0, str(PACKAGE_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))
os.chdir(NOTEBOOK_DIR)

import numpy as np
import torch
from nectary_env import FlowerParams
from nectary_allocentric_env import AllocentricNectarSearchEnv, CanvasParams
from training import _allocentric_vision_shape
from training_uncertainty_gated import (
    train_uncertainty_gated_allocentric_model, _allocentric_action_delta_bounds,
)
from uncertainty_gated_model import UncertaintyGatedAllocentricNectarySearchModel
from experiment_registry import load_pretrained_perception_components

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Load perception and select an environment

In [ ]:
PERCEPTION_CHECKPOINT = Path("checkpoints/perception.pt")
PRETRAIN_ENVIRONMENT = "flat-cross"
conditions = {
    "funnel-cross": (-1.0, "cross"),
    "funnel": (-1.0, "none"),
    "flat-cross": (-100.0, "cross"),
    "flat": (-100.0, "none"),
}
if not PERCEPTION_CHECKPOINT.is_file():
    raise FileNotFoundError("Run nectary.ipynb first, or set PERCEPTION_CHECKPOINT to its saved checkpoint.")
checkpoint = torch.load(PERCEPTION_CHECKPOINT, map_location=DEVICE)
model_config = checkpoint
c_value, pattern = conditions[PRETRAIN_ENVIRONMENT]

In [ ]:
flower_params = FlowerParams(
    r0= 2.0,
    R= 50.0,
    L= 25.0,
    c= -1.0,
)

canvas_params = CanvasParams(
    width= 200.0,
    height= 200.0,
    background_value= 0.,
    flower_value= 1.0,
    hole_value= 0.0,
    pattern_type= "cross",
    pattern_value= 0.5,
    cross_width= 5.0,
    stripe_width= 10.0,
    stripe_offset=5.0,
    stripe_orientation= "vertical",
    circular_radius= 10.0
)

In [ ]:
canvas_params.pattern_type = pattern
env = AllocentricNectarSearchEnv(
    flower=FlowerParams(r0=flower_params.r0, R=flower_params.R, L=flower_params.L, c=c_value),
    canvas=canvas_params,
    dt=0.01,
    max_steps=100,
    max_press_depth=5,
    visual_field_size=model_config["visual_field_size"],
)
action_delta_low, action_delta_high = _allocentric_action_delta_bounds(env)

## Initialize the exploration policy

In [ ]:
model = UncertaintyGatedAllocentricNectarySearchModel(
    mech_obs_dim=env.observation_space["mech"].shape[0],
    vis_obs_shape=_allocentric_vision_shape(env),
    action_dim=env.action_space.shape[0],
    mech_embedding_dim=model_config['mech_embedding_dim'],
    vis_embedding_dim=model_config['vis_embedding_dim'],
    fusion_dim=model_config['fusion_dim'],
    latent_dim=model_config['latent_dim'],
    hidden_dim=model_config['hidden_dim'],
    beta=model_config.get('beta',0.003),
    visual_likelihood=model_config['visual_likelihood'],
    learn_decoder_variance=model_config.get('learn_decoder_variance', False),
    posterior_fusion='moe',
    moment_matching=model_config.get('moment_matching', True),
    uncertainty_metric='joint',
    rnn_type='rnn',
    modality_rnn_mode=model_config.get('modality_rnn_mode', 'separate'),
    action_high=env.action_space.high,
    action_delta_low=action_delta_low,
    action_delta_high=action_delta_high,
    rl_algorithm='ppo',
    exploration_objective='realized_information_gain',
    gate_mode='fixed_threshold',
    fixed_gate_threshold=375,
    exploration_action_l2=0.,
    exploration_action_delta_huber_beta=1.0,
    action_selection_strategy='uncertainty_gated',
    information_gain_reward_coef=0.2
).to(DEVICE)

In [ ]:
initialization = load_pretrained_perception_components(model, PERCEPTION_CHECKPOINT, map_location=DEVICE)
print(f"Loaded {initialization['num_loaded_keys']} perception tensors.")

## Pretrain exploration

In [ ]:
trained_model, training_state = train_uncertainty_gated_allocentric_model(
    envs=[env],
    model=model,
    num_epochs=100,
    batch_size=20,
    seq_len=200,
    perception_learning_rate=0e-5,
    exploration_learning_rate=1e-4,
    exploitation_learning_rate=0.0,
    head_lr_scheduler=False,
    print_every=10,
    min_depth=0.,
    exploration_warmup_epochs=0,
    deterministic_exploitation=False,
    deterministic_exploration=False,
    detach_hidden_state=True,
    grad_clip_norm=None,
    metrics_csv_path=None,
    append_metrics_csv=False,
    parallel=False,
    device=DEVICE,
    force_mode='explore',
    return_training_state=True,
    gate_hysteresis=30,
    gate_uncertainty_ema_alpha=0.5
)

## Save the checkpoint for configuration-based experiments

In [ ]:
OUTPUT_CHECKPOINT = Path(f"checkpoints/exploration_{PRETRAIN_ENVIRONMENT}.pt")
OUTPUT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
checkpoint["model_state_dict"] = trained_model.state_dict()
checkpoint["exploration_pretraining"] = {
    "environment": PRETRAIN_ENVIRONMENT,
    "c_value": c_value,
    "pattern_type": pattern,
    "num_epochs": 200,
    "batch_size": 20,
    "seq_len": 200,
    "perception_learning_rate": 0.0,
    "exploration_learning_rate": 5e-5,
    "exploitation_learning_rate": 0.0,
    "force_mode": "explore",
}
torch.save(checkpoint, OUTPUT_CHECKPOINT)
print(f"Saved exploration checkpoint to {OUTPUT_CHECKPOINT}")